In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🎫 Support Ticket Knowledge Assistant

## What We're Building

An assistant that searches **past resolved support tickets** to suggest
likely fixes for new issues. When an agent gets a new ticket, the system:

1. Finds similar past tickets
2. Extracts the **resolution** from each match
3. Suggests a fix ranked by similarity and resolution success

## Why This is Hard for RAG

Support tickets are messy:
- Different people describe the same issue differently
- Resolutions are buried in long threads
- Some tickets are resolved but the fix didn't actually work

Our approach:
- Store **problem** and **resolution** separately as metadata
- Use **status** to weight successful resolutions higher
- **Semantic search** handles the "same issue, different words" problem

## Stack
- **LangChain** — orchestration
- **ChromaDB** — vector store with ticket metadata
- **Ollama** — local LLM + embeddings

## Step 1 — Imports

In [ ]:
# !pip install langchain langchain-ollama langchain-community chromadb -q

from datetime import datetime
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema import Document

print("All imports successful!")

## Step 2 — Sample Support Tickets

A set of resolved support tickets with structured fields.

In [ ]:
tickets = [
    {
        "id": "TK-1001",
        "subject": "Cannot login after password reset",
        "description": "Customer reports that after resetting their password via the forgot password flow, they still get 'Invalid credentials' error when trying to log in. They have tried clearing cookies and using incognito mode.",
        "category": "authentication",
        "priority": "high",
        "resolution": "The password reset email link was cached by the email client's link preview, which consumed the one-time token before the user clicked it. Solution: Generate a new reset link directly from the admin panel and send it via a fresh email. Also, a code fix was deployed to make reset tokens valid for 3 uses instead of 1.",
        "status": "resolved",
        "resolution_time_hours": 2,
        "date": "2024-11-15",
    },
    {
        "id": "TK-1002",
        "subject": "Dashboard loading blank page",
        "description": "User sees a white screen when accessing the analytics dashboard. Browser console shows 'ChunkLoadError: Loading chunk 14 failed'. Happens across Chrome and Firefox.",
        "category": "frontend",
        "priority": "high",
        "resolution": "Stale service worker was serving outdated JavaScript bundles. Fix: Add cache-busting headers to chunk files and instruct users to hard-refresh (Ctrl+Shift+R) or clear site data. Permanent fix: Service worker now checks for new versions every 5 minutes.",
        "status": "resolved",
        "resolution_time_hours": 4,
        "date": "2024-11-10",
    },
    {
        "id": "TK-1003",
        "subject": "API returns 500 on large CSV upload",
        "description": "When uploading a CSV file larger than 50MB through the data import API, the request fails with a 500 Internal Server Error. Smaller files (under 10MB) work fine. The error log shows 'ENOMEM: not enough memory'.",
        "category": "api",
        "priority": "medium",
        "resolution": "The API was loading the entire CSV into memory at once. Fix: Switch to streaming CSV parser that processes rows in chunks of 1000. Also increased the API gateway timeout from 30s to 120s for upload endpoints. The file size limit was documented as 200MB.",
        "status": "resolved",
        "resolution_time_hours": 6,
        "date": "2024-11-08",
    },
    {
        "id": "TK-1004",
        "subject": "Email notifications not being delivered",
        "description": "Multiple customers report not receiving notification emails for alerts they configured. The notification settings show as enabled. Webhook notifications work fine but email ones don't arrive.",
        "category": "notifications",
        "priority": "high",
        "resolution": "Our SendGrid API key had expired and the renewal wasn't applied to the production environment. The staging environment was working because it used a different key. Fix: Rotated the SendGrid API key, applied it to all environments through the secrets manager, and added monitoring for email delivery failures.",
        "status": "resolved",
        "resolution_time_hours": 1,
        "date": "2024-11-05",
    },
    {
        "id": "TK-1005",
        "subject": "Data export shows wrong timezone",
        "description": "When exporting data to CSV, all timestamps are shown in UTC even though the user's profile is set to US/Eastern. The dashboard shows correct times but the export doesn't.",
        "category": "data",
        "priority": "low",
        "resolution": "The export API was not reading the user's timezone preference from their profile. It was hardcoded to UTC. Fix: Modified the export endpoint to accept a 'timezone' query parameter and default to the user's profile timezone. The CSV header now includes the timezone used.",
        "status": "resolved",
        "resolution_time_hours": 3,
        "date": "2024-10-28",
    },
    {
        "id": "TK-1006",
        "subject": "Two-factor authentication code rejected",
        "description": "Customer's TOTP codes from Google Authenticator are being rejected. They confirmed the time on their phone is correct. The 2FA was set up three months ago and worked fine until today.",
        "category": "authentication",
        "priority": "high",
        "resolution": "The server's NTP sync had drifted by 45 seconds, causing TOTP validation window to miss. Fix: Re-synced the server clock with NTP. Also widened the TOTP acceptance window from ±1 step (30 seconds) to ±2 steps (60 seconds) to handle minor clock drift. Added NTP monitoring alert.",
        "status": "resolved",
        "resolution_time_hours": 1,
        "date": "2024-10-20",
    },
    {
        "id": "TK-1007",
        "subject": "Search returns no results for exact matches",
        "description": "Users report that searching for an exact product name like 'ProMax-2000' returns no results, even though the product exists. Partial searches like 'ProMax' also fail. General searches like 'premium product' work.",
        "category": "search",
        "priority": "medium",
        "resolution": "The search index was configured with aggressive tokenization that was splitting on hyphens and numbers. 'ProMax-2000' was indexed as ['promax', '2000'] but the query analyzer was treating it as a single token. Fix: Updated the search analyzer to handle hyphens and alphanumeric strings consistently. Rebuilt the search index.",
        "status": "resolved",
        "resolution_time_hours": 5,
        "date": "2024-10-15",
    },
    {
        "id": "TK-1008",
        "subject": "Billing shows duplicate charges",
        "description": "Customer is seeing two charges for the same subscription in their billing history. The amounts are identical and dated one minute apart. Their credit card was charged twice.",
        "category": "billing",
        "priority": "critical",
        "resolution": "A race condition in the payment processing webhook handler caused the subscription renewal to be processed twice when Stripe sent a retry. Fix: Added idempotency key check using Stripe's event ID. Refunded the duplicate charge and added a deduplication check that prevents processing the same webhook event ID twice.",
        "status": "resolved",
        "resolution_time_hours": 2,
        "date": "2024-10-10",
    },
    {
        "id": "TK-1009",
        "subject": "Mobile app crashes on image upload",
        "description": "iOS app crashes immediately when users try to upload a profile photo. Android version works. The crash happens before the upload starts, right when selecting the image from the gallery.",
        "category": "mobile",
        "priority": "high",
        "resolution": "The latest iOS update (17.2) changed the photo library permission model. Our app was using the old PHAuthorizationStatus API which crashes on iOS 17.2+. Fix: Updated to PHPickerViewController which doesn't require full photo library access. Released as hotfix v3.4.1.",
        "status": "resolved",
        "resolution_time_hours": 8,
        "date": "2024-10-05",
    },
    {
        "id": "TK-1010",
        "subject": "SSO integration failing with SAML error",
        "description": "Enterprise customer reports that SSO via Okta fails with 'Invalid SAML response'. They recently renewed their Okta IdP certificate. Other SSO providers (Azure AD) work fine.",
        "category": "authentication",
        "priority": "high",
        "resolution": "The customer regenerated their IdP signing certificate but didn't update it in our SSO settings. Our system was validating the SAML response against the old certificate. Fix: Updated the IdP certificate in the customer's SSO configuration. Added a more descriptive error message that says 'Certificate mismatch — please check if your IdP certificate was recently renewed'.",
        "status": "resolved",
        "resolution_time_hours": 1,
        "date": "2024-09-28",
    },
]

print(f"Loaded {len(tickets)} resolved support tickets")
for t in tickets:
    print(f"  [{t['id']}] {t['subject']} ({t['category']})")

## Step 3 — Index Tickets for Retrieval

We index the **problem description** for searching, but store the
**resolution** in metadata so we can display it alongside matches.

In [ ]:
def ticket_to_document(ticket: dict) -> Document:
    """
    Convert a ticket to a LangChain document.
    The page_content is the problem description (what we search by).
    The resolution is stored in metadata (what we display).
    """
    # Combine subject + description for richer embeddings
    content = f"Subject: {ticket['subject']}\n\nDescription: {ticket['description']}"
    
    return Document(
        page_content=content,
        metadata={
            "ticket_id": ticket["id"],
            "subject": ticket["subject"],
            "category": ticket["category"],
            "priority": ticket["priority"],
            "resolution": ticket["resolution"],
            "status": ticket["status"],
            "resolution_time_hours": ticket["resolution_time_hours"],
            "date": ticket["date"],
        },
    )


docs = [ticket_to_document(t) for t in tickets]

embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="support_tickets",
)

print(f"Indexed {vectorstore._collection.count()} tickets")

## Step 4 — New Ticket Resolution Suggester

When a new ticket comes in, we:
1. Find the 3 most similar past tickets
2. Show their resolutions
3. Ask the LLM to synthesize a suggested fix

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

suggest_prompt = ChatPromptTemplate.from_template(
    """You are a support ticket resolution assistant. A new support ticket has
been submitted, and I've found similar past tickets with their resolutions.

Analyze the similar tickets and suggest a resolution for the new ticket.

GUIDELINES:
- Start with the most likely fix based on the closest matching past ticket
- Note any differences between the past case and the new one
- Suggest diagnostic steps if the match isn't conclusive
- Rate your confidence: HIGH, MEDIUM, or LOW
- Include estimated resolution time based on past tickets

Similar Past Tickets:
{context}

New Ticket:
{new_ticket}

Suggested Resolution:"""
)


def suggest_resolution(subject: str, description: str) -> None:
    """Find similar tickets and suggest a resolution for a new issue."""
    print(f"🎫 NEW TICKET")
    print(f"   Subject: {subject}")
    print(f"   Description: {description}")
    print("=" * 60)
    
    # Search for similar tickets
    query = f"Subject: {subject}\n\nDescription: {description}"
    similar = vectorstore.similarity_search_with_score(query, k=3)
    
    # Display similar tickets
    print("\n📋 Similar Past Tickets:\n")
    context_parts = []
    for doc, score in similar:
        m = doc.metadata
        similarity = max(0, 1 - score)  # Convert distance to similarity
        print(f"  [{m['ticket_id']}] {m['subject']}")
        print(f"    Category: {m['category']} | Priority: {m['priority']}")
        print(f"    Resolved in: {m['resolution_time_hours']}h | Date: {m['date']}")
        print(f"    Resolution: {m['resolution'][:100]}...")
        print()
        
        context_parts.append(
            f"Ticket {m['ticket_id']} — {m['subject']}\n"
            f"Problem: {doc.page_content}\n"
            f"Resolution: {m['resolution']}\n"
            f"Category: {m['category']} | Resolved in: {m['resolution_time_hours']}h"
        )
    
    # Generate suggestion
    context = "\n\n---\n\n".join(context_parts)
    new_ticket = f"Subject: {subject}\nDescription: {description}"
    
    chain = suggest_prompt | llm | StrOutputParser()
    suggestion = chain.invoke({"context": context, "new_ticket": new_ticket})
    
    print("-" * 60)
    print(f"\n💡 SUGGESTED RESOLUTION:\n")
    print(suggestion)
    print("\n" + "=" * 60)


print("Support ticket assistant ready!")

## Step 5 — Test with New Tickets

In [ ]:
# Similar to TK-1001 (password reset) but with a twist
suggest_resolution(
    "Password reset link not working",
    "Customer clicked the password reset link in their email but gets "
    "'Token expired or invalid' error. They reset the password within "
    "5 minutes of receiving the email. Using Gmail on desktop."
)

In [ ]:
# Similar to TK-1003 (API errors) but different trigger
suggest_resolution(
    "API timeout on data sync",
    "Our scheduled data sync job that calls the bulk API endpoint is "
    "timing out after 30 seconds. The dataset has grown from 10K to "
    "500K records over the past month. It used to work fine."
)

In [ ]:
# Tests cross-category matching
suggest_resolution(
    "Users can't log in with Okta SSO",
    "Enterprise customer using Okta SSO reports all users getting a "
    "'Authentication failed' error as of this morning. No changes were "
    "made on our side. Their IT team says Okta is working for other apps."
)

In [ ]:
# A brand new issue with partial matches
suggest_resolution(
    "Dashboard charts showing stale data",
    "Charts on the analytics dashboard haven't updated in 3 days. The "
    "data API returns current data when queried directly, but the "
    "dashboard visualizations are stuck on old data."
)

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Problem-resolution split** | Index the problem for search, store resolution as metadata |
| **Similarity search + scores** | Rank past tickets by relevance to the new ticket |
| **Rich metadata** | Category, priority, resolution time — all searchable |
| **Resolution synthesis** | LLM adapts past resolutions to the specific new issue |
| **Confidence rating** | Prompt instructs LLM to rate its confidence |

## 🔧 Customization Ideas

- **Resolution feedback**: Track if suggestions actually worked → improve ranking
- **Category filtering**: Only search within the same product category
- **Time weighting**: Prefer recent tickets over old ones
- **Auto-routing**: Use the category prediction to route tickets to the right team